# Capítulo 3 — Embeddings (Information Embedding)

**Livro:** começa na página **124** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Measuring similarity between vectors · PDF p. 128

#### Exercício: Cosseno, produto escalar e distância

Para dois vetores de exemplo, calcule similaridade de cosseno, produto escalar e distância euclidiana. Depois normalize os vetores e mostre que o produto escalar passa a ser igual ao cosseno.

In [ ]:
a, b = np.array([3.0, 1.0, 2.0]), np.array([1.0, 0.5, 4.0])
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(round(cos, 4), a @ b, round(np.linalg.norm(a - b), 4))
an, bn = a / np.linalg.norm(a), b / np.linalg.norm(b); print(round(an @ bn, 4))

## Modern embedding models: OpenAI, jina, cohere, voyage · PDF p. 139

#### Exercício: Busca semântica com embeddings

Gere os embeddings dos 12 documentos com `embeddings()` e, para 3 perguntas escritas com palavras **diferentes** das do texto (ex.: "tem internet sem fio?"), mostre o documento mais similar.

In [ ]:
E = embeddings([d["titulo"] + ". " + d["texto"] for d in DOCS])
for q in ["tem internet sem fio?", "posso ir com meu cão-guia?", "até que horas fica aberta no domingo?"]:
    s = similaridade(E, embeddings(q)[0]); print(q, "->", DOCS[int(np.argmax(s))]["titulo"])

## Sparse embedding, dense embedding, and BM25 · PDF p. 149

#### Exercício: BM25 (esparso) × TF-IDF

Indexe os documentos com `rank_bm25.BM25Okapi` e com TF-IDF do scikit-learn (cosseno). Para "multa atraso bloqueio", mostre o top-3 de cada um.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
toks = [re.findall(r"\w+", d["texto"].lower()) for d in DOCS]
q = "multa atraso bloqueio"
bm = BM25Okapi(toks).get_scores(q.split())
vec = TfidfVectorizer(); M = vec.fit_transform(d["texto"] for d in DOCS)
tf = (M @ vec.transform([q]).T).toarray().ravel()
for nome, s in [("BM25", bm), ("TF-IDF", tf)]:
    print(nome, [DOCS[i]["titulo"] for i in np.argsort(-s)[:3]])